intro_to_Language_modeling part1

[link](https://youtu.be/TCH_1BHY58I)

Previous model was for predicting next char when a single char was given and we had only one layer.
Now we will improve the model to handle multiple input char(3 in this case) with multiple hidden layer

Goal: given 3 char we get the next most likely char

In [28]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [29]:
# download the training data
!curl https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt -o names.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  222k  100  222k    0     0   611k      0 --:--:-- --:--:-- --:--:--  612k


In [30]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

build dataset. i.e build x's and y's
-----


In [31]:
# build the vocabulary of characters and lookup mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [32]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:

  print(f'{w=}')
  context = [0] * block_size # create a list with len 3
  for ch in w + '.':
    ix = stoi[ch] # get char's index
    X.append(context) # initially it will be [0,0,0]
    Y.append(ix)
    #print(context, '--->', itos[ix])
    print('given '+''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append. eg [0,0] + [5] = [0,0,5] . where 5 is the index of char 'e'

X = torch.tensor(X) #Size [32, 3]
Y = torch.tensor(Y) #Size [32]

w='emma'
given ... ---> e
given ..e ---> m
given .em ---> m
given emm ---> a
given mma ---> .
w='olivia'
given ... ---> o
given ..o ---> l
given .ol ---> i
given oli ---> v
given liv ---> i
given ivi ---> a
given via ---> .
w='ava'
given ... ---> a
given ..a ---> v
given .av ---> a
given ava ---> .
w='isabella'
given ... ---> i
given ..i ---> s
given .is ---> a
given isa ---> b
given sab ---> e
given abe ---> l
given bel ---> l
given ell ---> a
given lla ---> .
w='sophia'
given ... ---> s
given ..s ---> o
given .so ---> p
given sop ---> h
given oph ---> i
given phi ---> a
given hia ---> .


In [33]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

Build a lookup table for input embeddings.
-----

using this table we could pick the corresponding embedding for the input char


In [34]:
# Here we are using 27X2, as there are 27 char and we going to represent each char with 2 random numbers
C = torch.randn((27, 2)) # this will be used as the input embedding lookup.

In [35]:
# so if we want to lookup for char 'e' whose index is 5 we can get embedding of it as following
C[5]

tensor([0.9078, 0.6374])

Embedding the inputs(i.e X)
-----

In [36]:
# suppose we want to get the embedding for a single item in X  ,say
X[1] # this would be [0, 0, 5] and which in character form would be [..e]

#embedding of X[1]
C[X[1]] # each 3 char of X(..e) is converted to its corresponding embeddings
#C[X[1]].shape # [3, 2]

tensor([[-0.4781, -0.3372],
        [-0.4781, -0.3372],
        [ 0.9078,  0.6374]])

In [37]:
# so embedding for the entire input X
emb = C[X]
emb.shape # shape [32, 3, 2].  32 inputs with 3 chars each and each char with two values

torch.Size([32, 3, 2])

Build hidden layer
-----

In [38]:
#random weights for 100 neurons(here we are just using 100 as an example)
W1 = torch.randn((6, 100)) # input to this layer is 6 bcoz, we have 3 inputs of 2 values each
#random biases for 100 neurons
b1 = torch.randn(100)

In [39]:
# next what we want is to do summazation (emb @ W1)+ b1
# First we convert the 'emb' to shape [32,6], bcoz we cannot matrix multiply 'emb'of shape [32, 3, 2] and weights W1 of shape [6, 100]

In [47]:
# Optional
emb[:,0,:] # all 32 input embeddings with first char out of the 3 chars
emb[:,1,:] # all 32 input embeddings with second char
emb[:,2,:] # all 32 input embeddings with third char

# now we can concat above 3 sequence of shape[32,2] to form a [32,6] shape tensor
torch.cat((emb[:,0,:], emb[:,1,:], emb[:,2,:]), 1).shape # here last arg 1 say to concat along the dim 1

# alternately, we can use torch.unbind instead of maually getting each seperate sequence as above
torch.unbind(emb, 1)
# and then concat
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])